# 08 · Multi-head latent attention (MLA): low-rank KV cache and matrix absorption

**Needed for:** DeepSeek-V2 MLA (LLM track, step 6), and every later DeepSeek model.

GQA shrinks the KV cache by sharing heads. MLA shrinks it differently: cache one small **latent vector**
$c_t$ per token (512 numbers in DeepSeek-V2) and reconstruct every head's keys and values from it with
up-projection matrices. Then, at inference, fold those up-projections into the query and output projections so
you never reconstruct K and V at all.

The catch is RoPE, which sits between the matrices and blocks the folding. MLA's answer is a small separate
"RoPE part" of each key, shared across heads.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. Low rank in one picture

A matrix $W \in \mathbb{R}^{m\times n}$ of rank $r$ factors as $W = UV$ with $U \in \mathbb{R}^{m\times r}$,
$V \in \mathbb{R}^{r\times n}$. Storing $Vx$ ($r$ numbers) instead of $Wx$ ($m$ numbers) is the whole MLA idea
for the cache. SVD gives the best rank-$r$ approximation of any matrix.

In [ ]:
m, n, true_rank = 256, 256, 32
W = torch.randn(m, true_rank) @ torch.randn(true_rank, n) + 0.01 * torch.randn(m, n)
U, S, Vh = torch.linalg.svd(W)
for r in (8, 32, 64):
    W_r = U[:, :r] @ torch.diag(S[:r]) @ Vh[:r]
    print(f"rank {r:3d}: relative error {((W - W_r).norm() / W.norm()).item():.4f}")

## 2. KV cache per token, per layer

| Scheme | Elements cached per token per layer |
|---|---|
| MHA | $2 \, n_h \, d_h$ |
| GQA | $2 \, n_{kv} \, d_h$ |
| MQA | $2 \, d_h$ |
| MLA | $d_c + d_h^R$ (latent + shared RoPE key) |

DeepSeek-V2 uses $n_h = 128$, $d_h = 128$, $d_c = 512$, $d_h^R = 64$.

In [ ]:
def kv_elements_per_token(scheme, n_heads=128, head_dim=128, n_kv_heads=8, d_c=512, d_rope=64):
    return {
        "MHA": 2 * n_heads * head_dim,
        "GQA": 2 * n_kv_heads * head_dim,
        "MQA": 2 * head_dim,
        "MLA": d_c + d_rope,
    }[scheme]


def test_kv_elements(fn):
    assert fn("MHA") == 32768
    assert fn("GQA", n_kv_heads=8) == 2048
    assert fn("MLA") == 576


test_kv_elements(kv_elements_per_token)
for s in ("MHA", "GQA", "MQA", "MLA"):
    e = kv_elements_per_token(s)
    print(f"{s}: {e:6d} elements = {e * 2 * 60 / 1024:6.1f} KiB per token for 60 layers in bf16")

## 3. MLA, the training-time (naive) form

Per token $x_t$:
- latent $c_t = W^{DKV} x_t$ ($d_c$), **cached**
- per-head content key $k^C_{t,h} = W^{UK}_h c_t$ and value $v_{t,h} = W^{UV}_h c_t$
- one RoPE key shared by all heads: $k^R_t = \text{RoPE}(W^{KR} x_t)$ ($d_h^R$), **cached**
- per-head query $q_{t,h} = [q^C_{t,h};\ \text{RoPE}(q^R_{t,h})]$
- key $k_{t,h} = [k^C_{t,h};\ k^R_t]$, then ordinary attention

(DeepSeek-V2 also compresses the queries through their own latent to save activation memory during training.
That's left out here because it doesn't change the cache.)

In [ ]:
def rope(x, pos, base=10000.0):
    """Half-split RoPE. x: (..., T, d), pos: (T,) positions."""
    d = x.shape[-1]
    inv = 1.0 / base ** (torch.arange(0, d, 2, dtype=x.dtype) / d)
    ang = pos[:, None].to(x.dtype) * inv
    cos, sin = torch.cat([ang, ang], -1).cos(), torch.cat([ang, ang], -1).sin()
    x1, x2 = x.chunk(2, -1)
    return x * cos + torch.cat([-x2, x1], -1) * sin


class MLA(nn.Module):
    def __init__(self, d_model=128, n_heads=4, d_nope=32, d_rope=16, d_v=32, d_c=48):
        super().__init__()
        self.H, self.dn, self.dr, self.dv, self.dc = n_heads, d_nope, d_rope, d_v, d_c
        self.W_dkv = nn.Linear(d_model, d_c, bias=False)
        self.W_uk = nn.Linear(d_c, n_heads * d_nope, bias=False)
        self.W_uv = nn.Linear(d_c, n_heads * d_v, bias=False)
        self.W_kr = nn.Linear(d_model, d_rope, bias=False)
        self.W_q = nn.Linear(d_model, n_heads * (d_nope + d_rope), bias=False)
        self.W_o = nn.Linear(n_heads * d_v, d_model, bias=False)
        self.scale = (d_nope + d_rope) ** -0.5

    def forward(self, x):
        """Naive form: rebuild full K and V for every head. x: (B, T, D)."""
        B, T, _ = x.shape
        pos = torch.arange(T)
        c = self.W_dkv(x)                                                     # (B, T, dc)
        k_nope = self.W_uk(c).view(B, T, self.H, self.dn).transpose(1, 2)     # (B, H, T, dn)
        v = self.W_uv(c).view(B, T, self.H, self.dv).transpose(1, 2)          # (B, H, T, dv)
        k_rope = rope(self.W_kr(x), pos)[:, None].expand(B, self.H, T, self.dr)
        q = self.W_q(x).view(B, T, self.H, self.dn + self.dr).transpose(1, 2)
        q_nope, q_rope = q.split([self.dn, self.dr], -1)
        q = torch.cat([q_nope, rope(q_rope, pos)], -1)
        k = torch.cat([k_nope, k_rope], -1)
        o = F.scaled_dot_product_attention(q, k, v, is_causal=True, scale=self.scale)
        return self.W_o(o.transpose(1, 2).reshape(B, T, self.H * self.dv))


mla = MLA().double()
x = torch.randn(2, 10, 128, dtype=torch.float64)
print(mla(x).shape)

## 4. Absorption: decoding straight from the cache

For the content part of the score:
$$q^{C\top}_h k^C_{j,h} = q^{C\top}_h W^{UK}_h c_j = \big(W^{UK\top}_h q^C_h\big)^\top c_j$$
So project the query into latent space once ($d_c$ numbers per head) and dot it with the cached $c_j$ directly.

For the output: $\sum_j p_j v_{j,h} = W^{UV}_h \sum_j p_j c_j$. Attend over the latents, then up-project once
(and $W^{UV}_h$ can be merged into $W^O$).

**Why RoPE blocks this:** with RoPE, $q^\top R_m^\top R_n W^{UK} c$ has a position-dependent rotation between
$q$ and $W^{UK}$, so no single fixed matrix can be pre-multiplied. That's why the RoPE part is a separate small
key computed from $x$, not from $c$.

In [ ]:
def mla_decode_absorbed(mla, x_new, c_cache, kr_cache, pos):
    """Attention output for ONE new token using only the caches.
    x_new: (B, D) new token; c_cache: (B, T, dc) latents incl. the new one; kr_cache: (B, T, dr) RoPE keys incl. new.
    pos: position of the new token (int)."""
    B = x_new.shape[0]
    H, dn, dr, dv, dc = mla.H, mla.dn, mla.dr, mla.dv, mla.dc
    q = mla.W_q(x_new).view(B, H, dn + dr)
    q_nope, q_rope = q.split([dn, dr], -1)
    q_rope = rope(q_rope[:, :, None], torch.tensor([pos]))[:, :, 0]            # (B, H, dr)

    W_uk = mla.W_uk.weight.view(H, dn, dc)                                     # per-head up-projection
    q_latent = torch.einsum("bhn,hnc->bhc", q_nope, W_uk)                      # absorbed query, (B, H, dc)
    scores = torch.einsum("bhc,btc->bht", q_latent, c_cache) + torch.einsum("bhr,btr->bht", q_rope, kr_cache)
    p = (scores * mla.scale).softmax(-1)
    ctx_latent = torch.einsum("bht,btc->bhc", p, c_cache)                      # attend over latents
    W_uv = mla.W_uv.weight.view(H, dv, dc)
    o = torch.einsum("bhc,hvc->bhv", ctx_latent, W_uv)                         # up-project once
    return mla.W_o(o.reshape(B, H * dv))


def test_absorbed(fn):
    torch.manual_seed(0)
    mla = MLA().double()
    x = torch.randn(2, 9, 128, dtype=torch.float64)
    full = mla(x)                                    # naive output at every position
    c_cache = mla.W_dkv(x)                           # what MLA actually caches
    kr_cache = rope(mla.W_kr(x), torch.arange(9))
    t = 8                                            # decode the last token
    out = fn(mla, x[:, t], c_cache[:, :t + 1], kr_cache[:, :t + 1], t)
    torch.testing.assert_close(out, full[:, t])


test_absorbed(mla_decode_absorbed)
print("absorbed decoding from the latent cache == naive attention")

## Things to be able to explain

- Absorbed decoding does its attention in $d_c = 512$ dimensions per head, more than $d_h = 128$. Why is it still
  faster for decoding? (Decoding is memory-bound: reading 576 cached numbers per token beats reading 32768.)
- Why is the naive form used for training and prefill? (There the work is compute-bound and the up-projected
  K/V are reused by many queries.)
- How does MLA compare with GQA at the same cache size? (DeepSeek-V2 reports better quality than MHA.)

## Drill

In [ ]:
def my_kv_elements_per_token(scheme, n_heads=128, head_dim=128, n_kv_heads=8, d_c=512, d_rope=64):
    raise NotImplementedError


def my_mla_decode_absorbed(mla, x_new, c_cache, kr_cache, pos):
    """Only use mla's weight matrices and the caches; never build per-head K or V."""
    raise NotImplementedError


drill(test_kv_elements, my_kv_elements_per_token)
drill(test_absorbed, my_mla_decode_absorbed)